In [10]:
import math
import numpy as np

for i in range(5):
    print(np.exp(i/2.2))

1.0
1.5754571033903182
2.482065084623012
3.9103870686464153
6.160647084304639


In [1]:
import pandas as pd

df=pd.DataFrame()
if df.columns.tolist():
    print('Exisst')

In [20]:
import yaml
with open('../data_schema/schema.yaml', 'rb') as f:
    schema=yaml.safe_load(f)
print(schema)
import pandas as pd
df = pd.read_csv('../artifacts/20_02_26_13_57_19/ingestion/split_data/train.csv')
print(len(schema['columns'])==len(df.columns.tolist()))
validated_all_columns=True
train_column_list=df.columns.tolist()
schema_column_list=[list(col.keys())[0] for col in schema['columns']]
print(train_column_list)
print(schema_column_list)
for col in schema_column_list:
    if col not in train_column_list:
        validated_all_columns=False
print(validated_all_columns)

{'columns': [{'site_name': 'str'}, {'log_date': 'date'}, {'cellid': 'int64'}, {'ueid': 'int64'}, {'uptime': 'int64'}, {'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2_ack': 'int64'}, {'rv2_nack': 'int64'}, {'rv2_dtx': 'int64'}, {'rv2_bler': 'float64'}, {'rv3_tx': 'int64'}, {'rv3_ack': 'int64'}, {'rv3_nack': 'int64'}, {'rv3_dtx': 'int64'}, {'rv3_bler': 'float64'}, {'rca_label': 'str'}], 'numerical_columns': [{'cellid': 'int64'}, {'ueid': 'int64'}, {'uptime': 'int64'}, {'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2

In [2]:
validated_num_of_columns=True
validate_all_columns=True
validated_num_of_columns and validate_all_columns

True

In [4]:
from scipy.stats import ks_2samp
import pandas as pd
import numpy as np
columns=['cqi', 'mcs', 'ri', 'ibler', 'rbler', 'resbler', 'tbler']
train_df = pd.read_csv('../artifacts/20_02_26_11_07_56/ingestion/split_data/train.csv')
test_df = pd.read_csv('../artifacts/20_02_26_11_07_56/ingestion/split_data/test.csv')
report = {}
for column in columns:
    report[column] = np.round(ks_2samp(train_df[column], test_df[column]).pvalue, 2)
print(report)

{'cqi': np.float64(0.61), 'mcs': np.float64(0.39), 'ri': np.float64(0.75), 'ibler': np.float64(0.8), 'rbler': np.float64(0.8), 'resbler': np.float64(0.9), 'tbler': np.float64(0.8)}


In [2]:
from databricks.sql import connect
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import yaml
load_dotenv()
query="""
SELECT *
FROM `du_stats`.`silver`.`synth_histo_table`
WHERE log_date=DATE('2026-01-01') AND ueid=17017
"""
with open('../data_schema/schema.yaml', 'r') as f:
    schema = yaml.safe_load(f)
print(schema)
columns = [list(col.keys())[0] for col in schema['columns']]
print(columns)
with connect(
    server_hostname=os.getenv('DATABRICKS_SERVER_HOSTNAME'),
    http_path=os.getenv('DATABRICKS_HTTP_PATH'),
    access_token=os.getenv('DATABRICKS_ACCESS_TOKEN')
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result=cursor.fetchall()
df = pd.DataFrame(data=result, columns=columns)
print(df.head())
import yaml
with open('../data_schema/schema.yaml','r') as f:
    numerical_schema = yaml.safe_load(f)
numerical_columns = [list(col.keys())[0] for col in numerical_schema['numerical_columns']]
df[numerical_columns].head()
arr = np.array(df)
print(arr[:5,:])

{'columns': [{'site_name': 'str'}, {'log_date': 'date'}, {'cellid': 'int64'}, {'ueid': 'int64'}, {'uptime': 'int64'}, {'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2_ack': 'int64'}, {'rv2_nack': 'int64'}, {'rv2_dtx': 'int64'}, {'rv2_bler': 'float64'}, {'rv3_tx': 'int64'}, {'rv3_ack': 'int64'}, {'rv3_nack': 'int64'}, {'rv3_dtx': 'int64'}, {'rv3_bler': 'float64'}, {'rca_label': 'str'}], 'numerical_columns': [{'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2_ack': 'int64'}, {'rv2_nack': 'int64'}, {'rv2_dtx': 'int64'},

In [4]:
import numpy as np
import pickle
with open('../artifacts/21_02_26_12_39_18/model_trainer/model/model.pkl', 'rb') as f:
    model=pickle.load(f)
np.round(model.feature_importances_, 2)

array([0.  , 0.12, 0.1 , 0.45, 0.  , 0.28, 0.03, 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ])